# NLP - Sentiment Analysis of IMDb Movie Reviews

I have created this notebook with ease of following along in mind - readability and clarity have taken priority over sheer performance on some occasions to make the logical flow of the task and my implementation of it as simple to follow as possible.

# Colab Drive Setup

First, I'll link my Google drive.

In [ ]:
# Colab drive setup

from google.colab import drive
from os import listdir
drive.mount('/content/drive')

Mounted at /content/drive


# Fetching the Reviews Data

Now I can fetch the reviews data (if you are following along with this, you will need the dataset uploaded to your Google Drive).

I support fetching the entire review, or alternatively fetching only the first 40 words of all the reviews isntead. The latter is commented out and was only used during development to decrease computation time.

In [ ]:
def fetch_reviews(pos_or_neg: str) -> dict:
  reviews = {}
  for file in listdir(f"/content/drive/MyDrive/Colab Notebooks/review_data/{pos_or_neg}"):
    with open(f"/content/drive/MyDrive/Colab Notebooks/review_data/{pos_or_neg}/{file}",'r') as review:

      if pos_or_neg == "pos":
        reviews[review.read()] = 1
      elif pos_or_neg == "neg":
        reviews[review.read()] = 0
      else:
        raise ValueError("Invalid class of review.")

      ###################

      # Alternatively I can use this block instead to use only 40 words of each review (saves on computation time for accuracy decrease - used in development for speedily adding changes)
      # review_text = review.read()

      # # store only the first 40 words
      # first_40_words = " ".join(review_text.split()[:40])

      # if pos_or_neg == "pos":
      #     reviews[first_40_words] = 1
      # elif pos_or_neg == "neg":
      #     reviews[first_40_words] = 0
      # else:
      #     raise ValueError("Invalid class of review.")

      ###################

  return reviews

raw_reviews_pos = fetch_reviews("pos")
print("First positive review:", list(raw_reviews_pos.keys())[0])

raw_reviews_neg = fetch_reviews("neg")
print("First negative review:", list(raw_reviews_neg.keys())[0])

First positive review: The film maybe goes a little far, but if you love the show it's what you expect. It's not a bad movie; it's actually pretty good. If you don't like the show, don't see the movie. It starts off a little slow maybe, but then picks up and turns out to be pretty funny. There are even a few "heart-wrenching" scenes toward the end. After all the protagonists have gone through these scences do get to you. Also Jerry throws in his opinion why his show upsets people and justifies his show's existience. He's got a pretty good point. We care so much about the private details of celebrities lives, so why is it wrong that these people tell their private lives on national TV, too. If they were celebrities we wouldn't mind at all, we'd eat it up. Do we not like his guest doing this just because they're poor white trash and it reminds us that there really is poverty in this world and not just rich glamous movie stars living in a "Leave it to Beaver" world?
First negative review:

# Create a Dataframe to store the reviews

I will be storing my reviews in a pandas DataFrame structure. This is flexible and compatible with other libraries, like being supported in test_train_split() for example.

In [ ]:
import pandas as pd

def create_dataframe(raw_reviews_pos: dict, raw_reviews_neg: dict) -> pd.DataFrame:
  combined_reviews = {**raw_reviews_pos, **raw_reviews_neg}
  df = pd.DataFrame(list(combined_reviews.items()), columns=["raw_review", "classification"])
  return df

df = create_dataframe(raw_reviews_pos, raw_reviews_neg)
df.tail()

,raw_review,classification
3994,Oh my. How can they make movies of such beauty...,0
3995,The main reason for writing this review is I f...,0
3996,What do you call a horror story without horror...,0
3997,This move was on TV last night. I guess as a t...,0
3998,I'm watching this on the Sci-Fi channel right ...,0


## Duplicate Review Present in the Dataset


"*Only to index 3998?! Where is the last review!?!"* - Since I used a dictionary to represent the data to assign classifications, a duplicate review caused there to be 1 less review in the set. I will leave this removed so I am only working with 3999 reviews.

# Splitting the Data

Now I can split the data into 3 sets using `train_test_split()` twice. I explain this process in more detail in the report, but essentially we end up with a balanced proportion of positive and negative examples in each of the sets, and the sets are in a 70:15:15 split.

In [ ]:
from sklearn.model_selection import train_test_split

# I have chosen a Train:Evaluation:Test ratio of 70:15:15

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=1, stratify=df['classification'])
eval_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=1, stratify=temp_df['classification'])

print("Train data shape:", train_df.shape)
print("Evaluation data shape:", eval_df.shape)
print("Test data shape:", test_df.shape)

print("\nClassification counts in TRAINING data:")
print(train_df['classification'].value_counts())

print("\nClassification counts in EVALUATION data:")
print(eval_df['classification'].value_counts())

print("\nClassification counts in TEST data:")
print(test_df['classification'].value_counts())

train_df.tail()

Train data shape: (2799, 2)
Evaluation data shape: (600, 2)
Test data shape: (600, 2)

Classification counts in TRAINING data:
classification
0    1400
1    1399
Name: count, dtype: int64

Classification counts in EVALUATION data:
classification
0    300
1    300
Name: count, dtype: int64

Classification counts in TEST data:
classification
1    300
0    300
Name: count, dtype: int64


,raw_review,classification
539,"Sure it was well shot and made, very well shot...",1
2135,"Nine minutes of psychedelic, pulsating, often ...",0
3426,Half the reviews were good so i took a chance ...,0
2369,"I wonder who, how and more importantly why the...",0
1988,One of the best if not the best rock'n'roll mo...,1


# Pre-processing Functionality

Here is some functionality for pre-processing, including text cleaning and tokensiation techniques. These will find use next.

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import LancasterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')

### --- DATA CLEANING --- ###

# Remove html break tags, punctuation, digits, etc
def clean_text(text: str) -> str:
  # Remove the br tags first before we remove any punctuation
  text = re.sub("<br />", "", text)

  # Now remove digits
  text = re.sub("[0-9]", "", text)

  # Now remove punctuation
  text = re.sub(r"[^\sa-zA-Z]", " ", text)

  # Make it all lowercase
  text = text.lower()

  return text

### --- TOKENISATION METHODS --- ###

# Tokenise by whitespace - assumes the data passed in is already cleaned
def tokenise_by_whitespace(text: str) -> list:
  return text.split()

# Stemming
def tokenise_by_stemming(text: str) -> list:

  # Get the stopwords using NLTK
  my_stopwords = set(stopwords.words('english'))

  # Init the stemmer
  stemmer = LancasterStemmer()

  # Tokenise, filter out stopwords, and apply stemming
  stemmed_words = [stemmer.stem(word) for word in word_tokenize(text.lower()) if word not in my_stopwords]

  return stemmed_words

# Lemmatising
def tokenise_by_lemmatising(text: str) -> list:

  # Get the stopwords using NLTK
  my_stopwords = set(stopwords.words('english'))

  # Init the lemmer
  lemmatiser = WordNetLemmatizer()

  # Tokenise, filter out stopwords, and apply lemmatisation
  lemmatised_words = [lemmatiser.lemmatize(word) for word in word_tokenize(text.lower()) if word not in my_stopwords]

  return lemmatised_words





[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


## Apply Tokenisation to the Train and Evaluation Set



I will tokenise all the reviews in the 3 different ways I support *now* and add them to the DataFrames. I will select which tokenisation to choose from and fetch later in my experiments. I went with this as I believe it is a clear way of showing which features I extract, when I do it, and what I do with them.

In [ ]:
train_df["cleaned_review"] = train_df["raw_review"].apply(clean_text)
train_df["tokenised_by_whitespace"] = train_df["cleaned_review"].apply(tokenise_by_whitespace)
train_df["tokenised_by_lemmatising"] = train_df["cleaned_review"].apply(tokenise_by_lemmatising)
train_df["tokenised_by_stemming"] = train_df["cleaned_review"].apply(tokenise_by_stemming)

train_df.tail()

,raw_review,classification,cleaned_review,tokenised_by_whitespace,tokenised_by_lemmatising,tokenised_by_stemming
539,"Sure it was well shot and made, very well shot...",1,sure it was well shot and made very well shot...,"[sure, it, was, well, shot, and, made, very, w...","[sure, well, shot, made, well, shot, made, sto...","[sur, wel, shot, mad, wel, shot, mad, story, w..."
2135,"Nine minutes of psychedelic, pulsating, often ...",0,nine minutes of psychedelic pulsating often ...,"[nine, minutes, of, psychedelic, pulsating, of...","[nine, minute, psychedelic, pulsating, often, ...","[nin, minut, psychedel, puls, oft, symmet, abs..."
3426,Half the reviews were good so i took a chance ...,0,half the reviews were good so i took a chance ...,"[half, the, reviews, were, good, so, i, took, ...","[half, review, good, took, chance, sure, prisc...","[half, review, good, took, chant, sur, priscil..."
2369,"I wonder who, how and more importantly why the...",0,i wonder who how and more importantly why the...,"[i, wonder, who, how, and, more, importantly, ...","[wonder, importantly, decision, call, richard,...","[wond, import, decid, cal, richard, attenborou..."
1988,One of the best if not the best rock'n'roll mo...,1,one of the best if not the best rock n roll mo...,"[one, of, the, best, if, not, the, best, rock,...","[one, best, best, rock, n, roll, movie, ever, ...","[on, best, best, rock, n, rol, movy, ev, mindl..."


In [ ]:
eval_df["cleaned_review"] = eval_df["raw_review"].apply(clean_text)
eval_df["tokenised_by_whitespace"] = eval_df["cleaned_review"].apply(tokenise_by_whitespace)
eval_df["tokenised_by_lemmatising"] = eval_df["cleaned_review"].apply(tokenise_by_lemmatising)
eval_df["tokenised_by_stemming"] = eval_df["cleaned_review"].apply(tokenise_by_stemming)

eval_df.tail()

,raw_review,classification,cleaned_review,tokenised_by_whitespace,tokenised_by_lemmatising,tokenised_by_stemming
1521,"I happened upon this film by accident, and rea...",1,i happened upon this film by accident and rea...,"[i, happened, upon, this, film, by, accident, ...","[happened, upon, film, accident, really, enjoy...","[hap, upon, film, accid, real, enjoy, timothy,..."
561,"I was really stunned how much a film, that's o...",1,i was really stunned how much a film that s o...,"[i, was, really, stunned, how, much, a, film, ...","[really, stunned, much, film, year, old, could...","[real, stun, much, film, year, old, could, imp..."
2883,"The words ""Swedish"" and ""Action movie"" do not ...",0,the words swedish and action movie do not ...,"[the, words, swedish, and, action, movie, do, ...","[word, swedish, action, movie, mix, becomes, o...","[word, swed, act, movy, mix, becom, obvy, ever..."
2842,The Japanese have always had incredible ambiti...,0,the japanese have always had incredible ambiti...,"[the, japanese, have, always, had, incredible,...","[japanese, always, incredible, ambition, fanta...","[japanes, alway, incred, ambit, fantasy, movy,..."
723,"I just viewed the film two days ago, and I was...",1,i just viewed the film two days ago and i was...,"[i, just, viewed, the, film, two, days, ago, a...","[viewed, film, two, day, ago, filled, anticipa...","[view, film, two, day, ago, fil, anticip, par,..."


# Frequency Distribution Usage to Drop Tokens

I have added the ability to drop a specified % of the most common and least common tokens that appear in some data. This will come in useful to save on runtime by reducing the number of features used to train the model. I chose this rather than choosing a flat singular value for flexbility and keep proportions more easy to follow and justifiable.

In [ ]:
import nltk
from nltk.corpus import stopwords

# I want to remove stop words and THEN remove the most frequent words.
def remove_stopwords(data: list) -> list:
    nltk_stopwords = set(stopwords.words('english'))
    filtered_reviews = [
        [token for token in review if token not in nltk_stopwords]
        for review in data
    ]
    return filtered_reviews

# Get the most and least common tokens to drop and return them
def get_most_and_least_freq_tokens(data: list, top_percent: float, bottom_percent: float) -> set:

    if top_percent + bottom_percent > 1:
        raise ValueError("Combined top and bottom proportions cannot be greater than 1.")

    # Combine all tokens
    all_tokens = [token for review in data for token in review]
    total_unique_tokens = len(set(all_tokens))

    # Create FreqDist over all tokens
    f_dist = nltk.FreqDist(all_tokens)

    # Sort in ascending order
    sorted_items = sorted(f_dist.items(), key=lambda x: x[1])

    # Cutoff index for list slicing
    top_cutoff_index = int(total_unique_tokens * (1 - top_percent))
    bottom_cutoff_index = int(total_unique_tokens * bottom_percent)

    tokens_to_drop = set()

    if (top_percent > 0 and top_percent <= 1):
        top_tokens = {token for token, _ in sorted_items[top_cutoff_index:]}
        print(f"{len(top_tokens)} of the most common tokens being dropped ({top_percent*100}% of all tokens)")
        tokens_to_drop = tokens_to_drop | top_tokens # combine
    if (bottom_percent > 0 and bottom_percent <= 1):
        bottom_tokens = {token for token, _ in sorted_items[:bottom_cutoff_index]}
        print(f"{len(bottom_tokens)} of the least common tokens being dropped ({bottom_percent*100}% of all tokens)")
        tokens_to_drop = tokens_to_drop | bottom_tokens # combine

    print(f"Total tokens to drop: {len(tokens_to_drop)}")
    return tokens_to_drop




# Filter the set of provided tokens from each review in the data
def filter_tokens(data: list, tokens_to_drop: set) -> list:
    filtered_reviews = [[token for token in review if token not in tokens_to_drop] for review in data]
    return filtered_reviews




In [ ]:
# Example use
train_tokenised_reviews = train_df["tokenised_by_stemming"].tolist()
tokens_to_drop = get_most_and_least_freq_tokens(train_tokenised_reviews, 0.003, 0.5)
print(tokens_to_drop)

# then I can use the filter_tokens() function to go through and remove the most common

52 of the most common tokens being dropped (0.3% of all tokens)
8513 of the least common tokens being dropped (50.0% of all tokens)
Total tokens to drop: 8565
{'baromet', 'wiz', 'fien', 'seep', 'swern', 'legrix', 'berrid', 'coleslaw', 'wimpy', 'puro', 'workplac', 'roi', 'peren', 'bilko', 'teru', 'kapl', 'oas', 'morbi', 'unsung', 'ger', 'radam', 'decree', 'unreleas', 'exult', 'infact', 'wlak', 'czechoslovak', 'withdrawn', 'jumpsuit', 'batho', 'mckenna', 'rickm', 'grumbl', 'hallway', 'ascertain', 'sexton', 'snarky', 'slith', 'boldest', 'sideb', 'unwash', 'kop', 'forebear', 'parol', 'bink', 'ev', 'whimp', 'roseann', 'teagu', 'mirn', 'kangwon', 'sweetin', 'chariot', 'ivanho', 'vray', 'pod', 'leev', 'paol', 'yoshinag', 'burnout', 'retin', 'uav', 'mcdougle', 'lockhee', 'doorbel', 'unspoil', 'ento', 'prochnow', 'gifford', 'characterizationrk', 'vag', 'heflin', 'grizzl', 'walkm', 'unshak', 'dogfight', 'flex', 'foley', 'exoskeleton', 'jaym', 'anglo', 'pero', 'walmington', 'disingenu', 'chesty',

# Extracting Compositional Phrases

## Finding n-grams

Here, I support finding ngrams in the reviews and I add these to their own column(s). I have chosen to add **all** of them to the dataframe - there are A LOT of them. I will, at a later point, extract only the most frequently occuring ones for use in my experiments and experiment with cutoff percentages.

In [ ]:
### --- Tokenisising reviews into lists of space separated ngrams --- ###

from nltk.util import ngrams

# Convert tokenized data containing reviews into a list of space-separated n-grams using the given n.
def get_ngrams(data: list, n: int) -> list:
  ngram_reviews = []
  for review in data:
      ngram_strings = [' '.join(ngram) for ngram in ngrams(review, n)]
      ngram_reviews.append(ngram_strings)
  return ngram_reviews



In [ ]:
### --- Now lets add the ngrams to their own columns for the TRAINING dataframe --- ###

# Get ngrams
tokenised_reviews = train_df["tokenised_by_whitespace"].tolist()
stopwords_removed = remove_stopwords(tokenised_reviews)

# Add them to the dataframe
train_df['bigrams'] = get_ngrams(stopwords_removed, 2)
train_df['trigrams'] = get_ngrams(stopwords_removed, 3)
train_df['quadgrams'] = get_ngrams(stopwords_removed, 4)
train_df['quintgrams'] = get_ngrams(stopwords_removed, 5)
# beyond n=5 didnt see improvements...


train_df.tail()

,raw_review,classification,cleaned_review,tokenised_by_whitespace,tokenised_by_lemmatising,tokenised_by_stemming,bigrams,trigrams,quadgrams,quintgrams
539,"Sure it was well shot and made, very well shot...",1,sure it was well shot and made very well shot...,"[sure, it, was, well, shot, and, made, very, w...","[sure, well, shot, made, well, shot, made, sto...","[sur, wel, shot, mad, wel, shot, mad, story, w...","[sure well, well shot, shot made, made well, w...","[sure well shot, well shot made, shot made wel...","[sure well shot made, well shot made well, sho...","[sure well shot made well, well shot made well..."
2135,"Nine minutes of psychedelic, pulsating, often ...",0,nine minutes of psychedelic pulsating often ...,"[nine, minutes, of, psychedelic, pulsating, of...","[nine, minute, psychedelic, pulsating, often, ...","[nin, minut, psychedel, puls, oft, symmet, abs...","[nine minutes, minutes psychedelic, psychedeli...","[nine minutes psychedelic, minutes psychedelic...","[nine minutes psychedelic pulsating, minutes p...","[nine minutes psychedelic pulsating often, min..."
3426,Half the reviews were good so i took a chance ...,0,half the reviews were good so i took a chance ...,"[half, the, reviews, were, good, so, i, took, ...","[half, review, good, took, chance, sure, prisc...","[half, review, good, took, chant, sur, priscil...","[half reviews, reviews good, good took, took c...","[half reviews good, reviews good took, good to...","[half reviews good took, reviews good took cha...","[half reviews good took chance, reviews good t..."
2369,"I wonder who, how and more importantly why the...",0,i wonder who how and more importantly why the...,"[i, wonder, who, how, and, more, importantly, ...","[wonder, importantly, decision, call, richard,...","[wond, import, decid, cal, richard, attenborou...","[wonder importantly, importantly decision, dec...","[wonder importantly decision, importantly deci...","[wonder importantly decision call, importantly...","[wonder importantly decision call richard, imp..."
1988,One of the best if not the best rock'n'roll mo...,1,one of the best if not the best rock n roll mo...,"[one, of, the, best, if, not, the, best, rock,...","[one, best, best, rock, n, roll, movie, ever, ...","[on, best, best, rock, n, rol, movy, ev, mindl...","[one best, best best, best rock, rock n, n rol...","[one best best, best best rock, best rock n, r...","[one best best rock, best best rock n, best ro...","[one best best rock n, best best rock n roll, ..."


In [ ]:
### --- Now lets add the ngrams to their own columns for the EVALUATION dataframe --- ###

# Get ngrams
tokenised_reviews = eval_df["tokenised_by_whitespace"].tolist()
stopwords_removed = remove_stopwords(tokenised_reviews)

# Add them to the dataframe
eval_df['bigrams'] = get_ngrams(stopwords_removed, 2)
eval_df['trigrams'] = get_ngrams(stopwords_removed, 3)
eval_df['quadgrams'] = get_ngrams(stopwords_removed, 4)
eval_df['quintgrams'] = get_ngrams(stopwords_removed, 5)


eval_df.tail()

,raw_review,classification,cleaned_review,tokenised_by_whitespace,tokenised_by_lemmatising,tokenised_by_stemming,bigrams,trigrams,quadgrams,quintgrams
1521,"I happened upon this film by accident, and rea...",1,i happened upon this film by accident and rea...,"[i, happened, upon, this, film, by, accident, ...","[happened, upon, film, accident, really, enjoy...","[hap, upon, film, accid, real, enjoy, timothy,...","[happened upon, upon film, film accident, acci...","[happened upon film, upon film accident, film ...","[happened upon film accident, upon film accide...","[happened upon film accident really, upon film..."
561,"I was really stunned how much a film, that's o...",1,i was really stunned how much a film that s o...,"[i, was, really, stunned, how, much, a, film, ...","[really, stunned, much, film, year, old, could...","[real, stun, much, film, year, old, could, imp...","[really stunned, stunned much, much film, film...","[really stunned much, stunned much film, much ...","[really stunned much film, stunned much film y...","[really stunned much film years, stunned much ..."
2883,"The words ""Swedish"" and ""Action movie"" do not ...",0,the words swedish and action movie do not ...,"[the, words, swedish, and, action, movie, do, ...","[word, swedish, action, movie, mix, becomes, o...","[word, swed, act, movy, mix, becom, obvy, ever...","[words swedish, swedish action, action movie, ...","[words swedish action, swedish action movie, a...","[words swedish action movie, swedish action mo...","[words swedish action movie mix, swedish actio..."
2842,The Japanese have always had incredible ambiti...,0,the japanese have always had incredible ambiti...,"[the, japanese, have, always, had, incredible,...","[japanese, always, incredible, ambition, fanta...","[japanes, alway, incred, ambit, fantasy, movy,...","[japanese always, always incredible, incredibl...","[japanese always incredible, always incredible...","[japanese always incredible ambitions, always ...","[japanese always incredible ambitions fantasy,..."
723,"I just viewed the film two days ago, and I was...",1,i just viewed the film two days ago and i was...,"[i, just, viewed, the, film, two, days, ago, a...","[viewed, film, two, day, ago, filled, anticipa...","[view, film, two, day, ago, fil, anticip, par,...","[viewed film, film two, two days, days ago, ag...","[viewed film two, film two days, two days ago,...","[viewed film two days, film two days ago, two ...","[viewed film two days ago, film two days ago f..."


## PoS and Consituency Parsing for Noun Phrases and Verb Phrases

I have used spaCy to extract noun and verb phrases within a certain length range (default is between 2 and 5 as this seemed appropriate). I have added this to the dataframes' columns. Again, I will extract only the common ones at a later point in experiments.

In [ ]:
import spacy

# Load model
nlp = spacy.load("en_core_web_sm")

# Extracts noun and verb phrases from text (given the phrase is in the provided length range)
def extract_np_vp(text, min_len=2, max_len=5):
    doc = nlp(text)
    phrases = []

    # Extract noun phrases
    for np in doc.noun_chunks:
        np_text = np.text
        word_count = len(np_text.split())
        if min_len <= word_count <= max_len:
            phrases.append(np_text)

    # Extract verb phrases
    for token in doc:
        if token.pos_ == "VERB":
            vp = " ".join([child.text for child in token.subtree])
            word_count = len(vp.split())
            if min_len <= word_count <= max_len:
                phrases.append(vp)

    return phrases

In [ ]:
# Get NPs and VPs for the training dataframe
train_df['noun_verb_phrases'] = train_df['cleaned_review'].apply(extract_np_vp)

# and get NPs and VPs for the evaluation dataframe
eval_df['noun_verb_phrases'] = eval_df['cleaned_review'].apply(extract_np_vp)

eval_df.tail()

,raw_review,classification,cleaned_review,tokenised_by_whitespace,tokenised_by_lemmatising,tokenised_by_stemming,bigrams,trigrams,quadgrams,quintgrams,noun_verb_phrases
1521,"I happened upon this film by accident, and rea...",1,i happened upon this film by accident and rea...,"[i, happened, upon, this, film, by, accident, ...","[happened, upon, film, accident, really, enjoy...","[hap, upon, film, accid, real, enjoy, timothy,...","[happened upon, upon film, film accident, acci...","[happened upon film, upon film accident, film ...","[happened upon film accident, upon film accide...","[happened upon film accident really, upon film...","[this film, timothy busfield s character, one ..."
561,"I was really stunned how much a film, that's o...",1,i was really stunned how much a film that s o...,"[i, was, really, stunned, how, much, a, film, ...","[really, stunned, much, film, year, old, could...","[real, stun, much, film, year, old, could, imp...","[really stunned, stunned much, much film, film...","[really stunned much, stunned much film, much ...","[really stunned much film, stunned much film y...","[really stunned much film years, stunned much ...","[how much a film, no stars, the realism, the f..."
2883,"The words ""Swedish"" and ""Action movie"" do not ...",0,the words swedish and action movie do not ...,"[the, words, swedish, and, action, movie, do, ...","[word, swedish, action, movie, mix, becomes, o...","[word, swed, act, movy, mix, becom, obvy, ever...","[words swedish, swedish action, action movie, ...","[words swedish action, swedish action movie, a...","[words swedish action movie, swedish action mo...","[words swedish action movie mix, swedish actio...","[the words, swedish and action movie, every ..."
2842,The Japanese have always had incredible ambiti...,0,the japanese have always had incredible ambiti...,"[the, japanese, have, always, had, incredible,...","[japanese, always, incredible, ambition, fanta...","[japanes, alway, incred, ambit, fantasy, movy,...","[japanese always, always incredible, incredibl...","[japanese always incredible, always incredible...","[japanese always incredible ambitions, always ...","[japanese always incredible ambitions fantasy,...","[the japanese, incredible ambitions, their fan..."
723,"I just viewed the film two days ago, and I was...",1,i just viewed the film two days ago and i was...,"[i, just, viewed, the, film, two, days, ago, a...","[viewed, film, two, day, ago, filled, anticipa...","[view, film, two, day, ago, fil, anticip, par,...","[viewed film, film two, two days, days ago, ag...","[viewed film two, film two days, two days ago,...","[viewed film two days, film two days ago, two ...","[viewed film two days ago, film two days ago f...","[the film, that paris, my second favorite city..."


# Normalise

## TF-IDF

In [ ]:
# The basis for this code was mainly inspired from the week 3 lab, but I have made
# some adjustments for clarity and ease of following along.

import math
import numpy as np


def get_term_frequencies(data: list) -> list:
    """
    Get the term frequencies for each review in the provided data.

    Returns a list of dictionaries - one per review, containing the term frequencies
    for each term in the review.
    """

    tf_full_list = []

    # Loop through all the reviews and the term count for each token that
    #  appears in the review.
    for review in data:
        tf_dict = {}
        for token in review:
            if token in tf_dict:
                tf_dict[token] += 1
            else:
                tf_dict[token] = 1

        tf_full_list.append(tf_dict)

    # Return the resulting list of dicts that was built up
    return tf_full_list

def get_vocabulary(data: list) -> set:
    """
    Return a sorted set of all unique terms in all the provided data.
    """

    # Just loop through each token in each review in our data and add it to the set.
    # Usage of the set data structure will take care of any duplicates for us.
    vocab = set()
    for review in data:
        for token in review:
            vocab.add(token)
    return sorted(vocab)

def get_inverse_document_frequencies(data: list, vocabulary: set) -> dict:
    """
    Get the IDFs (inverse doc frequencies) for each term in the provided vocabulary.

    Returns a dictionary mapping the terms in the vocabulary to their IDFs.
    """

    # Number of total reviews is required in the formula I use later
    n = len(data)

    # Now for each term in my vocab, I want to calculate the DF (doc frequency).
    # The DF essentially says 'how many documents does this term occur in?'.
    # The IDF is given by the formula:
    # IDF(term) = log10(N / (DF(term) + 1) )  - +1 is to avoid div by 0
    doc_idfs = {}
    for term in vocabulary:
        doc_freq = 0
        for review in data:
            if term in review:
                doc_freq += 1

        # Update the IDFs dictionary - a map of terms to IDF values
        doc_idfs[term] = math.log(float(n)/float(doc_freq + 1), 10)

    return doc_idfs

def get_tf_idfs(tfs: list, idfs: dict) -> list:
    """
    Takes in the term frequencies and document frequencies, and uses them to
    calculate TF-IDF values.

    Returns a list of dictionaries containing the terms (keys) in each doc along
    with its TF-IDF (values).

    NOTE:
    - tfs is a list of dictionaries containing the terms (keys) in a document
      along with their frequency (values).
    - idfs is a dictionary containing all the terms in the vocabulary (keys)
      along with their IDF (values).
    """

    # Loop through all the TF dictionary for each doc (review) in our tf list
    tf_idfs = []
    for doc in tfs:
        tf_idf_doc = {}
        for term, tf_value in doc.items():
            # Handle the case where the term isnt in idfs
            idf_value = idfs.get(term, 0)

            # TF-IDF is simply given by multiplying TF by IDF
            tf_idf_doc[term] = tf_value * idf_value

        tf_idfs.append(tf_idf_doc)

    return tf_idfs

def vectorise_tf_idfs(tf_idfs: list, vocabulary: set) -> list:
    """
    Convert the TF-IDF scores from being in a list of dictionaries to instead be
    represented by a numpy 2D matrix, where:
    - Each row is a doc (review)
    - Each column is a term in the vocab

    Returns a numpy matrix of TF-IDF values, of shape (num_docs, num_terms).
    """

    # Initialise a np matrix of the correct dims
    num_docs = len(tf_idfs)
    num_terms = len(vocabulary)
    tf_idf_matrix = np.zeros((num_docs, num_terms))

    # Fill out with values
    # Loop through each document's TF-IDF value dictionary and keep track of its
    # index to update in the np matrix
    for doc_index, tf_idf_dict in enumerate(tf_idfs):
        # Now loop through the terms and TF-IDF pairs in each doc
        for term, tf_idf_val in tf_idf_dict.items():
            # And for each term, we only add it to the matrix if the term is in
            # the provided vocab
            if term in vocabulary:
                term_index = vocabulary.index(term)
                tf_idf_matrix[doc_index, term_index] = tf_idf_val

    return tf_idf_matrix

## L2 Normalisation

This will be discussed more in the report, but essentially I added this so longer reviews dont have higher term frequencies simply because they are longer reviews...

In [ ]:
import numpy as np

def l2_normalise_tfs(tfs: list):
    """
    Takes in term frequncies for each doc, and returns an updated list of L2
    normalised term frequencies.

    Returns a list of dictionaries containing the terms (keys) in each doc along
    with the L2 normalised TF-IDFs (values).
    """

    l2_normalised_data = []

    # I loop through each of the tf dictionaries for each doc, and normalise
    # each of the TF entries using L2 normalisation
    for doc in tfs:
        # Handle 0-length case
        if len(doc) == 0:
            l2_normalised_data.append(doc)
            continue

        # L2 norm is given by the square root of the sum of all TFs in that doc
        sum_of_squares = sum(tf ** 2 for tf in doc.values())
        l2_norm = math.sqrt(sum_of_squares)

        # Create a new dict with the updated normalised TF value and add it to
        # the ongoing list
        normalized_tf_dict = {term: value / l2_norm for term, value in doc.items()}
        l2_normalised_data.append(normalized_tf_dict)

    return l2_normalised_data

# Features Selection

Here is how I will be experimenting with my features. Let's run through the general setup of what I need:

My dataframes are in the following format:

```raw_review | classification | t_by_whitespace (all) | t_by_lemm (all) | t_by_stem (all) | bigrams (all) | trigrams (all)```

I should:
- take my singular tokens (via whitespace,lemmatising, or stemming)
- optionally (and experimentally...):
  - do a freq dist and remove the most and least common tokens - cutoff to be experimented with
  - take my bigrams and trigrams, and filter the top N out
  - take nounphrases, and filter the top N out
- combine whichever I select into my final feature set: e.g, tokenised + bigrams + trigrams + ...
- then do TF-IDF calculations on these

### Predict with `MultinomialNB` and Show Results

I'll be using the following function to run my predictions through the model and get results. This will use TF-IDF. In the experiments in this notebook, I only included results from using L2 normalisation. I did do experiments without L2 but these gave worse accuracies - they are shown in my supporting report.

In [ ]:
### --- To run the predictions and show accuracy tests using TF-IDF --- ###

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

def run_model_predictions(final_train_reviews: list, final_eval_reviews: list, vc: set, is_test_set: bool):
    # Train set
    train_tf = get_term_frequencies(final_train_reviews)
    train_tf = l2_normalise_tfs(train_tf)
    idf = get_inverse_document_frequencies(final_train_reviews, vc)
    train_tf_idf = get_tf_idfs(train_tf, idf)
    vectorised_train_tf_idfs = vectorise_tf_idfs(train_tf_idf, vc)

    print("Vectorised Train TF-IDFs Shape:", vectorised_train_tf_idfs.shape)

    # Eval set
    eval_tf = get_term_frequencies(final_eval_reviews)
    eval_tf = l2_normalise_tfs(eval_tf)
    eval_tf_idf = get_tf_idfs(eval_tf, idf)
    vectorised_eval_tf_idfs = vectorise_tf_idfs(eval_tf_idf, vc)

    print("Vectorised Evaluation TF-IDFs Shape:", vectorised_eval_tf_idfs.shape)

    ### --- Now train model and get predictions --- ###

    nb_classifier = MultinomialNB()
    nb_classifier.fit(vectorised_train_tf_idfs, train_df["classification"]) # train
    eval_predictions = nb_classifier.predict(vectorised_eval_tf_idfs) # predict

    # Which dataframe I use depends on whether I am doing predictions for the test set or the evaluation set - hence the bool parameter needed
    if is_test_set:
        true_labels = test_df["classification"] # what should have I predicted?
    else:
        true_labels = eval_df["classification"] # what should have I predicted?

    # See results
    print("Accuracy:", accuracy_score(true_labels, eval_predictions))
    print("Classification Report Summary:")
    print(classification_report(true_labels, eval_predictions))

## Whitespace Tokenisation Only

Let's start basic - only use tokenisation using the whitespace tokens I made earlier, without any stopwords removed. I'll drop some tokens based on frequency distributions.

In [ ]:
# Token top and bottom %s to remove
TOP_TOKEN_PROPORTION = 0.01 # Aims to remove stopwords and also most common tokens
BOTTOM_TOKEN_PROPORTION = 0.60
TOKENISATION_TO_USE = "tokenised_by_whitespace"

### --- Tokenising and filtering the train dataset --- ###

# Get singular tokens and filter
train_tokenised_reviews = train_df[TOKENISATION_TO_USE].tolist()
tokens_to_drop = get_most_and_least_freq_tokens(train_tokenised_reviews, TOP_TOKEN_PROPORTION, BOTTOM_TOKEN_PROPORTION)
train_tokens_filtered = filter_tokens(train_tokenised_reviews, tokens_to_drop)

# Now combine all remaining tokens
final_train_reviews = train_tokens_filtered # in this case, it is simply our singular tokens. for the next tests I will add to these with bigrams, etc
print(f"First review in train set: {final_train_reviews[0]}")

### --- Tokenising the evaluation dataset --- ###

# Get singular tokens
eval_tokenised_reviews = eval_df[TOKENISATION_TO_USE].tolist()

# Now combine all remaining tokens
final_eval_reviews = eval_tokenised_reviews # in this case, it is simply our singular tokens. for the next tests I will add to these with bigrams, etc
print(f"First review in evaluation set: {final_eval_reviews[0]}")

### --- Get the Vocabulary --- ###

vc = get_vocabulary(final_train_reviews)
print(f"\nThe vocabulary, {len(vc)} in length: ")
print(vc)

304 of the most common tokens being dropped (1.0% of all tokens)
18201 of the least common tokens being dropped (60.0% of all tokens)
Total tokens to drop: 18505
First review in train set: ['unfortunate', 'leading', 'comments', 'include', 'words', 'clueless', 'appalling', 'nonsense', 'excellent', 'entertainment', 'suspend', 'disbelief', 'homosexual', 'lesbian', 'fall', 'child', 'live', 'together', 'happily', 'wonderful', 'heart', 'impossible', 'implausible', 'events', 'described', 'negative', 'comment', 'ended', 'drives', 'initial', 'house', 'explore', 'whether', 'retains', 'homosexuality']
First review in evaluation set: ['i', 'feel', 'totally', 'ripped', 'off', 'someone', 'needs', 'to', 'refund', 'the', 'i', 'spent', 'at', 'blockbuster', 'to', 'rent', 'this', 'homemade', 'mess', 'this', 'is', 'not', 'a', 'musical', 'it', 'is', 'a', 'complete', 'waste', 'of', 'time', 'and', 'my', 'evening', 'what', 'i', 'don', 't', 'get', 'is', 'why', 'did', 'this', 'get', 'distributed', 'in', 'the', 

In [ ]:
run_model_predictions(final_train_reviews, final_eval_reviews, vc, False)

Vectorised Train TF-IDFs Shape: (2799, 11830)
Vectorised Evaluation TF-IDFs Shape: (600, 11830)
Accuracy: 0.8116666666666666
Classification Report Summary:
              precision    recall  f1-score   support

           0       0.78      0.87      0.82       300
           1       0.85      0.75      0.80       300

    accuracy                           0.81       600
   macro avg       0.82      0.81      0.81       600
weighted avg       0.82      0.81      0.81       600



## Lemmatisation Only

In [ ]:
# Token top and bottom %s to remove
TOP_TOKEN_PROPORTION = 0.001
BOTTOM_TOKEN_PROPORTION = 0.60
TOKENISATION_TO_USE = "tokenised_by_lemmatising"

### --- Tokenising and filtering the train dataset --- ###

# Get singular tokens and filter
train_tokenised_reviews = train_df[TOKENISATION_TO_USE].tolist()
tokens_to_drop = get_most_and_least_freq_tokens(train_tokenised_reviews, TOP_TOKEN_PROPORTION, BOTTOM_TOKEN_PROPORTION)
train_tokens_filtered = filter_tokens(train_tokenised_reviews, tokens_to_drop)

# Now combine all remaining tokens
final_train_reviews = train_tokens_filtered # in this case, it is simply our singular tokens. for the next tests I will add to these with bigrams, etc
print(f"First review in train set: {final_train_reviews[0]}")

### --- Tokenising the evaluation set --- ###

# Get singular tokens
eval_tokenised_reviews = eval_df[TOKENISATION_TO_USE].tolist()

# Now combine all remaining tokens
final_eval_reviews = eval_tokenised_reviews # in this case, it is simply our singular tokens. for the next tests I will add to these with bigrams, etc
print(f"First review in evaluation set: {final_eval_reviews[0]}")

### --- Get the Vocabulary --- ###

vc = get_vocabulary(final_train_reviews)
print(f"\nThe vocabulary, {len(vc)} in length: ")
print(vc)

27 of the most common tokens being dropped (0.1% of all tokens)
16198 of the least common tokens being dropped (60.0% of all tokens)
Total tokens to drop: 16225
First review in train set: ['think', 'unfortunate', 'leading', 'comment', 'include', 'word', 'clueless', 'appalling', 'nonsense', 'think', 'funny', 'excellent', 'entertainment', 'suspend', 'disbelief', 'homosexual', 'man', 'lesbian', 'woman', 'fall', 'love', 'child', 'live', 'together', 'happily', 'ever', 'always', 'wonderful', 'played', 'heart', 'impossible', 'far', 'implausible', 'event', 'described', 'acting', 'script', 'funny', 'negative', 'comment', 'ended', 'family', 'drive', 'away', 'initial', 'house', 'instead', 'explore', 'whether', 'man', 'retains', 'homosexuality']
First review in evaluation set: ['feel', 'totally', 'ripped', 'someone', 'need', 'refund', 'spent', 'blockbuster', 'rent', 'homemade', 'mess', 'musical', 'complete', 'waste', 'time', 'evening', 'get', 'get', 'distributed', 'first', 'place', 'somebody', 'mu

In [ ]:
run_model_predictions(final_train_reviews, final_eval_reviews, vc, False)

Vectorised Train TF-IDFs Shape: (2799, 10772)
Vectorised Evaluation TF-IDFs Shape: (600, 10772)
Accuracy: 0.825
Classification Report Summary:
              precision    recall  f1-score   support

           0       0.79      0.88      0.83       300
           1       0.87      0.77      0.81       300

    accuracy                           0.82       600
   macro avg       0.83      0.82      0.82       600
weighted avg       0.83      0.82      0.82       600



## Stemming Only

In [ ]:
# Token top and bottom %s to remove
TOP_TOKEN_PROPORTION = 0.001
BOTTOM_TOKEN_PROPORTION = 0.60
TOKENISATION_TO_USE = "tokenised_by_stemming"

### --- Tokenising and filtering the train dataset --- ###

# Get singular tokens and filter
train_tokenised_reviews = train_df[TOKENISATION_TO_USE].tolist()
tokens_to_drop = get_most_and_least_freq_tokens(train_tokenised_reviews, TOP_TOKEN_PROPORTION, BOTTOM_TOKEN_PROPORTION)
train_tokens_filtered = filter_tokens(train_tokenised_reviews, tokens_to_drop)

# Now combine all remaining tokens
final_train_reviews = train_tokens_filtered # in this case, it is simply our singular tokens. for the next tests I will add to these with bigrams, etc
print(f"First review in train set: {final_train_reviews[0]}")

### --- Tokenising the evaluation dataset --- ###

# Get singular tokens
eval_tokenised_reviews = eval_df[TOKENISATION_TO_USE].tolist()

# Now combine all remaining tokens
final_eval_reviews = eval_tokenised_reviews # in this case, it is simply our singular tokens. for the next tests I will add to these with bigrams, etc
print(f"First review in evaluation set: {final_eval_reviews[0]}")

### --- Get the Vocabulary --- ###

vc = get_vocabulary(final_train_reviews)
print(f"\nThe vocabulary, {len(vc)} in length: ")
print(vc)

18 of the most common tokens being dropped (0.1% of all tokens)
10216 of the least common tokens being dropped (60.0% of all tokens)
Total tokens to drop: 10234
First review in train set: ['think', 'unfortun', 'lead', 'includ', 'word', 'clueless', 'appal', 'nonsens', 'think', 'funny', 'excel', 'entertain', 'suspend', 'disbeliev', 'homosex', 'man', 'lesb', 'wom', 'could', 'fal', 'lov', 'child', 'liv', 'togeth', 'happy', 'alway', 'wond', 'play', 'heart', 'warm', 'imposs', 'far', 'implaus', 'describ', 'script', 'funny', 'neg', 'could', 'wel', 'end', 'famy', 'driv', 'away', 'init', 'hous', 'instead', 'extend', 'expl', 'wheth', 'man', 'retain', 'resid', 'homosex']
First review in evaluation set: ['feel', 'tot', 'rip', 'someon', 'nee', 'refund', 'spent', 'blockbust', 'rent', 'homemad', 'mess', 'mus', 'complet', 'wast', 'tim', 'ev', 'get', 'get', 'distribut', 'first', 'plac', 'somebody', 'must', 'heavy', 'drug', 'night', 'deal', 'mad', 'seen', 'bet', 'film', 'com', 'film', 'schools', 'film', 

In [ ]:
run_model_predictions(final_train_reviews, final_eval_reviews, vc, False)

Vectorised Train TF-IDFs Shape: (2799, 6793)
Vectorised Evaluation TF-IDFs Shape: (600, 6793)
Accuracy: 0.8183333333333334
Classification Report Summary:
              precision    recall  f1-score   support

           0       0.78      0.88      0.83       300
           1       0.86      0.76      0.81       300

    accuracy                           0.82       600
   macro avg       0.82      0.82      0.82       600
weighted avg       0.82      0.82      0.82       600



## Stemming + bigrams

In [ ]:
# Token top and bottom %s to remove
TOP_TOKEN_PROPORTION = 0.001
BOTTOM_TOKEN_PROPORTION = 0.6
BOTTOM_NGRAMS_PROPORTION = 0.99
TOKENISATION_TO_USE = "tokenised_by_stemming"

### --- Tokenising and filtering the train dataset --- ###

# Get singular tokens and filter
train_tokenised_reviews = train_df[TOKENISATION_TO_USE].tolist()
tokens_to_drop = get_most_and_least_freq_tokens(train_tokenised_reviews, TOP_TOKEN_PROPORTION, BOTTOM_TOKEN_PROPORTION)
train_tokens_filtered = filter_tokens(train_tokenised_reviews, tokens_to_drop)

# Now I can do the same for bigrams
train_bigrams = train_df["bigrams"].tolist()
bigrams_to_drop = get_most_and_least_freq_tokens(train_bigrams, 0, BOTTOM_NGRAMS_PROPORTION)
train_bigrams_filtered = filter_tokens(train_bigrams, bigrams_to_drop)

# Now combine all remaining tokens
final_train_reviews = [
    tokens + bigrams
    for tokens, bigrams in zip(train_tokens_filtered, train_bigrams_filtered)
]
print(f"First review in train set: {final_train_reviews[0]}")

### --- Tokenising for evaluation set ###

# Get singular tokens
eval_tokenised_reviews = eval_df[TOKENISATION_TO_USE].tolist()

# Same for bigrams
eval_bigrams = eval_df["bigrams"].tolist()

# Now combine all remaining tokens
final_eval_reviews = [
    tokens + bigrams
    for tokens, bigrams in zip(eval_tokenised_reviews, eval_bigrams)
]
print(f"First review in evaluation set: {final_eval_reviews[0]}")

### --- Get the Vocabulary --- ###

vc = get_vocabulary(final_train_reviews)
print(f"\nThe vocabulary, {len(vc)} in length: ")
print(vc)

18 of the most common tokens being dropped (0.1% of all tokens)
10216 of the least common tokens being dropped (60.0% of all tokens)
Total tokens to drop: 10234
273385 of the least common tokens being dropped (99.0% of all tokens)
Total tokens to drop: 273385
First review in train set: ['think', 'unfortun', 'lead', 'includ', 'word', 'clueless', 'appal', 'nonsens', 'think', 'funny', 'excel', 'entertain', 'suspend', 'disbeliev', 'homosex', 'man', 'lesb', 'wom', 'could', 'fal', 'lov', 'child', 'liv', 'togeth', 'happy', 'alway', 'wond', 'play', 'heart', 'warm', 'imposs', 'far', 'implaus', 'describ', 'script', 'funny', 'neg', 'could', 'wel', 'end', 'famy', 'driv', 'away', 'init', 'hous', 'instead', 'extend', 'expl', 'wheth', 'man', 'retain', 'resid', 'homosex', 'funny movie', 'fall love', 'movie one', 'acting good', 'good script', 'story could', 'could well']
First review in evaluation set: ['feel', 'tot', 'rip', 'someon', 'nee', 'refund', 'spent', 'blockbust', 'rent', 'homemad', 'mess', 'm

In [ ]:
run_model_predictions(final_train_reviews, final_eval_reviews, vc, False)

Vectorised Train TF-IDFs Shape: (2799, 9555)
Vectorised Evaluation TF-IDFs Shape: (600, 9555)
Accuracy: 0.8366666666666667
Classification Report Summary:
              precision    recall  f1-score   support

           0       0.80      0.89      0.85       300
           1       0.88      0.78      0.83       300

    accuracy                           0.84       600
   macro avg       0.84      0.84      0.84       600
weighted avg       0.84      0.84      0.84       600



## Lemmatisation + bigrams

In [ ]:
# Token top and bottom %s to remove
TOP_TOKEN_PROPORTION = 0.001
BOTTOM_TOKEN_PROPORTION = 0.6
BOTTOM_NGRAMS_PROPORTION = 0.99
TOKENISATION_TO_USE = "tokenised_by_lemmatising"

### --- Tokenising and filtering the train dataset --- ###

# Get singular tokens and filter
train_tokenised_reviews = train_df[TOKENISATION_TO_USE].tolist()
tokens_to_drop = get_most_and_least_freq_tokens(train_tokenised_reviews, TOP_TOKEN_PROPORTION, BOTTOM_TOKEN_PROPORTION)
train_tokens_filtered = filter_tokens(train_tokenised_reviews, tokens_to_drop)

# Now I can do the same for bigrams
train_bigrams = train_df["bigrams"].tolist()
bigrams_to_drop = get_most_and_least_freq_tokens(train_bigrams, 0, BOTTOM_NGRAMS_PROPORTION)
train_bigrams_filtered = filter_tokens(train_bigrams, bigrams_to_drop)

# Now combine all remaining tokens
final_train_reviews = [
    tokens + bigrams
    for tokens, bigrams in zip(train_tokens_filtered, train_bigrams_filtered)
]
print(f"First review in train set: {final_train_reviews[0]}")

### --- Tokenising for evaluation set ###

# Get singular tokens
eval_tokenised_reviews = eval_df[TOKENISATION_TO_USE].tolist()

# Same for bigrams
eval_bigrams = eval_df["bigrams"].tolist()

# Now combine all remaining tokens
final_eval_reviews = [
    tokens + bigrams
    for tokens, bigrams in zip(eval_tokenised_reviews, eval_bigrams)
]
print(f"First review in evaluation set: {final_eval_reviews[0]}")

### --- Get the Vocabulary --- ###

vc = get_vocabulary(final_train_reviews)
print(f"\nThe vocabulary, {len(vc)} in length: ")
print(vc)

27 of the most common tokens being dropped (0.1% of all tokens)
16198 of the least common tokens being dropped (60.0% of all tokens)
Total tokens to drop: 16225
273385 of the least common tokens being dropped (99.0% of all tokens)
Total tokens to drop: 273385
First review in train set: ['think', 'unfortunate', 'leading', 'comment', 'include', 'word', 'clueless', 'appalling', 'nonsense', 'think', 'funny', 'excellent', 'entertainment', 'suspend', 'disbelief', 'homosexual', 'man', 'lesbian', 'woman', 'fall', 'love', 'child', 'live', 'together', 'happily', 'ever', 'always', 'wonderful', 'played', 'heart', 'impossible', 'far', 'implausible', 'event', 'described', 'acting', 'script', 'funny', 'negative', 'comment', 'ended', 'family', 'drive', 'away', 'initial', 'house', 'instead', 'explore', 'whether', 'man', 'retains', 'homosexuality', 'funny movie', 'fall love', 'movie one', 'acting good', 'good script', 'story could', 'could well']
First review in evaluation set: ['feel', 'totally', 'ripp

In [ ]:
run_model_predictions(final_train_reviews, final_eval_reviews, vc, False)

Vectorised Train TF-IDFs Shape: (2799, 13534)
Vectorised Evaluation TF-IDFs Shape: (600, 13534)
Accuracy: 0.8283333333333334
Classification Report Summary:
              precision    recall  f1-score   support

           0       0.80      0.88      0.84       300
           1       0.87      0.78      0.82       300

    accuracy                           0.83       600
   macro avg       0.83      0.83      0.83       600
weighted avg       0.83      0.83      0.83       600



## Stemming + bigrams + trigrams

In [ ]:
# Token top and bottom %s to remove
TOP_TOKEN_PROPORTION = 0.001
BOTTOM_TOKEN_PROPORTION = 0.6
BOTTOM_BIGRAMS_PROPORTION = 0.99
BOTTOM_TRIGRAMS_PROPORTION = 0.999
TOKENISATION_TO_USE = "tokenised_by_stemming"

### --- Tokenising and filtering the train dataset --- ###

# Get singular tokens and filter
train_tokenised_reviews = train_df[TOKENISATION_TO_USE].tolist()
tokens_to_drop = get_most_and_least_freq_tokens(train_tokenised_reviews, TOP_TOKEN_PROPORTION, BOTTOM_TOKEN_PROPORTION)
train_tokens_filtered = filter_tokens(train_tokenised_reviews, tokens_to_drop)

# Now I can do the same for bigrams...
train_bigrams = train_df["bigrams"].tolist()
bigrams_to_drop = get_most_and_least_freq_tokens(train_bigrams, 0, BOTTOM_BIGRAMS_PROPORTION)
train_bigrams_filtered = filter_tokens(train_bigrams, bigrams_to_drop)

# ... and trigrams
train_trigrams = train_df["trigrams"].tolist()
trigrams_to_drop = get_most_and_least_freq_tokens(train_trigrams, 0, BOTTOM_TRIGRAMS_PROPORTION)
train_trigrams_filtered = filter_tokens(train_trigrams, trigrams_to_drop)

# Now combine all remaining tokens
final_train_reviews = [
    tokens + bigrams + trigrams
    for tokens, bigrams, trigrams in zip(train_tokens_filtered, train_bigrams_filtered, train_trigrams_filtered)
]
print(f"First review in train set: {final_train_reviews[0]}")

### --- Tokenising the evaluation dataset --- ###

# Get singular tokens and filter (use filter tokens received from train sett)
eval_tokenised_reviews = eval_df[TOKENISATION_TO_USE].tolist()

# Same for bigrams...
eval_bigrams = eval_df["bigrams"].tolist()

# and trigrams
eval_trigrams = eval_df["trigrams"].tolist()

# Now combine all remaining tokens
final_eval_reviews = [
    tokens + bigrams + trigrams
    for tokens, bigrams, trigrams in zip(eval_tokenised_reviews, eval_bigrams, eval_trigrams)
]
print(f"First review in evaluation set: {final_eval_reviews[0]}")

### --- Get the Vocabulary --- ###

vc = get_vocabulary(final_train_reviews)
print(f"\nThe vocabulary, {len(vc)} in length: ")
print(vc)

18 of the most common tokens being dropped (0.1% of all tokens)
10216 of the least common tokens being dropped (60.0% of all tokens)
Total tokens to drop: 10234
273385 of the least common tokens being dropped (99.0% of all tokens)
Total tokens to drop: 273385
337593 of the least common tokens being dropped (99.9% of all tokens)
Total tokens to drop: 337593
First review in train set: ['think', 'unfortun', 'lead', 'includ', 'word', 'clueless', 'appal', 'nonsens', 'think', 'funny', 'excel', 'entertain', 'suspend', 'disbeliev', 'homosex', 'man', 'lesb', 'wom', 'could', 'fal', 'lov', 'child', 'liv', 'togeth', 'happy', 'alway', 'wond', 'play', 'heart', 'warm', 'imposs', 'far', 'implaus', 'describ', 'script', 'funny', 'neg', 'could', 'wel', 'end', 'famy', 'driv', 'away', 'init', 'hous', 'instead', 'extend', 'expl', 'wheth', 'man', 'retain', 'resid', 'homosex', 'funny movie', 'fall love', 'movie one', 'acting good', 'good script', 'story could', 'could well']
First review in evaluation set: ['

In [ ]:
run_model_predictions(final_train_reviews, final_eval_reviews, vc, False)

Vectorised Train TF-IDFs Shape: (2799, 9893)
Vectorised Evaluation TF-IDFs Shape: (600, 9893)
Accuracy: 0.8333333333333334
Classification Report Summary:
              precision    recall  f1-score   support

           0       0.80      0.89      0.84       300
           1       0.87      0.78      0.82       300

    accuracy                           0.83       600
   macro avg       0.84      0.83      0.83       600
weighted avg       0.84      0.83      0.83       600



## Lemmatisation + bigrams + trigrams

In [ ]:
# Token top and bottom %s to remove
TOP_TOKEN_PROPORTION = 0.001
BOTTOM_TOKEN_PROPORTION = 0.6
BOTTOM_BIGRAMS_PROPORTION = 0.99
BOTTOM_TRIGRAMS_PROPORTION = 0.999
TOKENISATION_TO_USE = "tokenised_by_lemmatising"

### --- Tokenising and filtering the train dataset --- ###

# Get singular tokens and filter
train_tokenised_reviews = train_df[TOKENISATION_TO_USE].tolist()
tokens_to_drop = get_most_and_least_freq_tokens(train_tokenised_reviews, TOP_TOKEN_PROPORTION, BOTTOM_TOKEN_PROPORTION)
train_tokens_filtered = filter_tokens(train_tokenised_reviews, tokens_to_drop)

# Now I can do the same for bigrams...
train_bigrams = train_df["bigrams"].tolist()
bigrams_to_drop = get_most_and_least_freq_tokens(train_bigrams, 0, BOTTOM_BIGRAMS_PROPORTION)
train_bigrams_filtered = filter_tokens(train_bigrams, bigrams_to_drop)

# ... and trigrams
train_trigrams = train_df["trigrams"].tolist()
trigrams_to_drop = get_most_and_least_freq_tokens(train_trigrams, 0, BOTTOM_TRIGRAMS_PROPORTION)
train_trigrams_filtered = filter_tokens(train_trigrams, trigrams_to_drop)

# Now combine all remaining tokens
final_train_reviews = [
    tokens + bigrams + trigrams
    for tokens, bigrams, trigrams in zip(train_tokens_filtered, train_bigrams_filtered, train_trigrams_filtered)
]
print(f"First review in train set: {final_train_reviews[0]}")

### --- Tokenising the evaluation dataset --- ###

# Get singular tokens and filter (use filter tokens received from train sett)
eval_tokenised_reviews = eval_df[TOKENISATION_TO_USE].tolist()

# Same for bigrams...
eval_bigrams = eval_df["bigrams"].tolist()

# and trigrams
eval_trigrams = eval_df["trigrams"].tolist()

# Now combine all remaining tokens
final_eval_reviews = [
    tokens + bigrams + trigrams
    for tokens, bigrams, trigrams in zip(eval_tokenised_reviews, eval_bigrams, eval_trigrams)
]
print(f"First review in evaluation set: {final_eval_reviews[0]}")

### --- Get the Vocabulary --- ###

vc = get_vocabulary(final_train_reviews)
print(f"\nThe vocabulary, {len(vc)} in length: ")
print(vc)

27 of the most common tokens being dropped (0.1% of all tokens)
16198 of the least common tokens being dropped (60.0% of all tokens)
Total tokens to drop: 16225
273385 of the least common tokens being dropped (99.0% of all tokens)
Total tokens to drop: 273385
337593 of the least common tokens being dropped (99.9% of all tokens)
Total tokens to drop: 337593
First review in train set: ['think', 'unfortunate', 'leading', 'comment', 'include', 'word', 'clueless', 'appalling', 'nonsense', 'think', 'funny', 'excellent', 'entertainment', 'suspend', 'disbelief', 'homosexual', 'man', 'lesbian', 'woman', 'fall', 'love', 'child', 'live', 'together', 'happily', 'ever', 'always', 'wonderful', 'played', 'heart', 'impossible', 'far', 'implausible', 'event', 'described', 'acting', 'script', 'funny', 'negative', 'comment', 'ended', 'family', 'drive', 'away', 'initial', 'house', 'instead', 'explore', 'whether', 'man', 'retains', 'homosexuality', 'funny movie', 'fall love', 'movie one', 'acting good', 'g

In [ ]:
run_model_predictions(final_train_reviews, final_eval_reviews, vc, False)

Vectorised Train TF-IDFs Shape: (2799, 13872)
Vectorised Evaluation TF-IDFs Shape: (600, 13872)
Accuracy: 0.8283333333333334
Classification Report Summary:
              precision    recall  f1-score   support

           0       0.80      0.88      0.84       300
           1       0.86      0.78      0.82       300

    accuracy                           0.83       600
   macro avg       0.83      0.83      0.83       600
weighted avg       0.83      0.83      0.83       600



## Stemming + bigrams + trigrams + quadgrams

In [ ]:
# Token top and bottom %s to remove
TOP_TOKEN_PROPORTION = 0.001
BOTTOM_TOKEN_PROPORTION = 0.60
BOTTOM_BIGRAMS_PROPORTION = 0.99
BOTTOM_TRIGRAMS_PROPORTION = 0.999
BOTTOM_QUADGRAMS_PROPORTION = 0.9999
TOKENISATION_TO_USE = "tokenised_by_stemming"

### --- Tokenising and filtering the train dataset --- ###

# Get singular tokens and filter
train_tokenised_reviews = train_df[TOKENISATION_TO_USE].tolist()
tokens_to_drop = get_most_and_least_freq_tokens(train_tokenised_reviews, TOP_TOKEN_PROPORTION, BOTTOM_TOKEN_PROPORTION)
train_tokens_filtered = filter_tokens(train_tokenised_reviews, tokens_to_drop)

# Now I can do the same for bigrams...
train_bigrams = train_df["bigrams"].tolist()
bigrams_to_drop = get_most_and_least_freq_tokens(train_bigrams, 0, BOTTOM_BIGRAMS_PROPORTION)
train_bigrams_filtered = filter_tokens(train_bigrams, bigrams_to_drop)

# ... and trigrams
train_trigrams = train_df["trigrams"].tolist()
trigrams_to_drop = get_most_and_least_freq_tokens(train_trigrams, 0, BOTTOM_TRIGRAMS_PROPORTION)
train_trigrams_filtered = filter_tokens(train_trigrams, trigrams_to_drop)

# ... and quadgrams
train_quadgrams = train_df["quadgrams"].tolist()
quadgrams_to_drop = get_most_and_least_freq_tokens(train_quadgrams, 0, BOTTOM_QUADGRAMS_PROPORTION)
train_quadgrams_filtered = filter_tokens(train_quadgrams, quadgrams_to_drop)

# Now combine all remaining tokens
final_train_reviews = [
    tokens + bigrams + trigrams + quadgrams
    for tokens, bigrams, trigrams, quadgrams in zip(train_tokens_filtered, train_bigrams_filtered, train_trigrams_filtered, train_quadgrams_filtered)
]
print(f"First review in train set: {final_train_reviews[0]}")

### --- Tokenising evaluation dataset --- ###

# Get singular tokens
eval_tokenised_reviews = eval_df[TOKENISATION_TO_USE].tolist()

# Same for bigrams...
eval_bigrams = eval_df["bigrams"].tolist()

# and trigrams
eval_trigrams = eval_df["trigrams"].tolist()

# and quadgrams
eval_quadgrams = eval_df["quadgrams"].tolist()

# Now combine all remaining tokens
final_eval_reviews = [
    tokens + bigrams + trigrams + quadgrams
    for tokens, bigrams, trigrams, quadgrams in zip(eval_tokenised_reviews, eval_bigrams, eval_trigrams, eval_quadgrams)
]
print(f"First review in evaluation set: {final_eval_reviews[0]}")

### --- Get the Vocabulary --- ###

vc = get_vocabulary(final_train_reviews)
print(f"\nThe vocabulary, {len(vc)} in length: ")
print(vc)

18 of the most common tokens being dropped (0.1% of all tokens)
10216 of the least common tokens being dropped (60.0% of all tokens)
Total tokens to drop: 10234
273385 of the least common tokens being dropped (99.0% of all tokens)
Total tokens to drop: 273385
337593 of the least common tokens being dropped (99.9% of all tokens)
Total tokens to drop: 337593
339280 of the least common tokens being dropped (99.99% of all tokens)
Total tokens to drop: 339280
First review in train set: ['think', 'unfortun', 'lead', 'includ', 'word', 'clueless', 'appal', 'nonsens', 'think', 'funny', 'excel', 'entertain', 'suspend', 'disbeliev', 'homosex', 'man', 'lesb', 'wom', 'could', 'fal', 'lov', 'child', 'liv', 'togeth', 'happy', 'alway', 'wond', 'play', 'heart', 'warm', 'imposs', 'far', 'implaus', 'describ', 'script', 'funny', 'neg', 'could', 'wel', 'end', 'famy', 'driv', 'away', 'init', 'hous', 'instead', 'extend', 'expl', 'wheth', 'man', 'retain', 'resid', 'homosex', 'funny movie', 'fall love', 'movie

In [ ]:
run_model_predictions(final_train_reviews, final_eval_reviews, vc, False)

Vectorised Train TF-IDFs Shape: (2799, 9927)
Vectorised Evaluation TF-IDFs Shape: (600, 9927)
Accuracy: 0.8333333333333334
Classification Report Summary:
              precision    recall  f1-score   support

           0       0.80      0.89      0.84       300
           1       0.87      0.78      0.82       300

    accuracy                           0.83       600
   macro avg       0.84      0.83      0.83       600
weighted avg       0.84      0.83      0.83       600



## Lemmatisation + bigrams + trigrams + quadgrams

In [ ]:
# Token top and bottom %s to remove
TOP_TOKEN_PROPORTION = 0.001
BOTTOM_TOKEN_PROPORTION = 0.60
BOTTOM_BIGRAMS_PROPORTION = 0.99
BOTTOM_TRIGRAMS_PROPORTION = 0.999
BOTTOM_QUADGRAMS_PROPORTION = 0.9999
TOKENISATION_TO_USE = "tokenised_by_lemmatising"

### --- Tokenising and filtering the train dataset --- ###

# Get singular tokens and filter
train_tokenised_reviews = train_df[TOKENISATION_TO_USE].tolist()
tokens_to_drop = get_most_and_least_freq_tokens(train_tokenised_reviews, TOP_TOKEN_PROPORTION, BOTTOM_TOKEN_PROPORTION)
train_tokens_filtered = filter_tokens(train_tokenised_reviews, tokens_to_drop)

# Now I can do the same for bigrams...
train_bigrams = train_df["bigrams"].tolist()
bigrams_to_drop = get_most_and_least_freq_tokens(train_bigrams, 0, BOTTOM_BIGRAMS_PROPORTION)
train_bigrams_filtered = filter_tokens(train_bigrams, bigrams_to_drop)

# ... and trigrams
train_trigrams = train_df["trigrams"].tolist()
trigrams_to_drop = get_most_and_least_freq_tokens(train_trigrams, 0, BOTTOM_TRIGRAMS_PROPORTION)
train_trigrams_filtered = filter_tokens(train_trigrams, trigrams_to_drop)

# ... and quadgrams
train_quadgrams = train_df["quadgrams"].tolist()
quadgrams_to_drop = get_most_and_least_freq_tokens(train_quadgrams, 0, BOTTOM_QUADGRAMS_PROPORTION)
train_quadgrams_filtered = filter_tokens(train_quadgrams, quadgrams_to_drop)

# Now combine all remaining tokens
final_train_reviews = [
    tokens + bigrams + trigrams + quadgrams
    for tokens, bigrams, trigrams, quadgrams in zip(train_tokens_filtered, train_bigrams_filtered, train_trigrams_filtered, train_quadgrams_filtered)
]
print(f"First review in train set: {final_train_reviews[0]}")

### --- Tokenising evaluation dataset --- ###

# Get singular tokens
eval_tokenised_reviews = eval_df[TOKENISATION_TO_USE].tolist()

# Same for bigrams...
eval_bigrams = eval_df["bigrams"].tolist()

# and trigrams
eval_trigrams = eval_df["trigrams"].tolist()

# and quadgrams
eval_quadgrams = eval_df["quadgrams"].tolist()

# Now combine all remaining tokens
final_eval_reviews = [
    tokens + bigrams + trigrams + quadgrams
    for tokens, bigrams, trigrams, quadgrams in zip(eval_tokenised_reviews, eval_bigrams, eval_trigrams, eval_quadgrams)
]
print(f"First review in evaluation set: {final_eval_reviews[0]}")

### --- Get the Vocabulary --- ###

vc = get_vocabulary(final_train_reviews)
print(f"\nThe vocabulary, {len(vc)} in length: ")
print(vc)

27 of the most common tokens being dropped (0.1% of all tokens)
16198 of the least common tokens being dropped (60.0% of all tokens)
Total tokens to drop: 16225
273385 of the least common tokens being dropped (99.0% of all tokens)
Total tokens to drop: 273385
337593 of the least common tokens being dropped (99.9% of all tokens)
Total tokens to drop: 337593
339280 of the least common tokens being dropped (99.99% of all tokens)
Total tokens to drop: 339280
First review in train set: ['think', 'unfortunate', 'leading', 'comment', 'include', 'word', 'clueless', 'appalling', 'nonsense', 'think', 'funny', 'excellent', 'entertainment', 'suspend', 'disbelief', 'homosexual', 'man', 'lesbian', 'woman', 'fall', 'love', 'child', 'live', 'together', 'happily', 'ever', 'always', 'wonderful', 'played', 'heart', 'impossible', 'far', 'implausible', 'event', 'described', 'acting', 'script', 'funny', 'negative', 'comment', 'ended', 'family', 'drive', 'away', 'initial', 'house', 'instead', 'explore', 'whe

In [ ]:
run_model_predictions(final_train_reviews, final_eval_reviews, vc, False)

Vectorised Train TF-IDFs Shape: (2799, 13906)
Vectorised Evaluation TF-IDFs Shape: (600, 13906)
Accuracy: 0.8283333333333334
Classification Report Summary:
              precision    recall  f1-score   support

           0       0.80      0.88      0.84       300
           1       0.86      0.78      0.82       300

    accuracy                           0.83       600
   macro avg       0.83      0.83      0.83       600
weighted avg       0.83      0.83      0.83       600



Quintgrams and beyond I did not test, as quadgrams were not effective in improving the accuracy in my experiments using L2 normalisation.

## Stemming + Noun Phrases and Verb Phrases


In [ ]:
# Token top and bottom %s to remove
TOP_TOKEN_PROPORTION = 0.001
BOTTOM_TOKEN_PROPORTION = 0.6
BOTTOM_NP_VP_PROPORTION = 0.995
TOKENISATION_TO_USE = "tokenised_by_stemming"

### --- Tokenising and filtering the train dataset --- ###

# Get singular tokens and filter
train_tokenised_reviews = train_df[TOKENISATION_TO_USE].tolist()
tokens_to_drop = get_most_and_least_freq_tokens(train_tokenised_reviews, TOP_TOKEN_PROPORTION, BOTTOM_TOKEN_PROPORTION)
train_tokens_filtered = filter_tokens(train_tokenised_reviews, tokens_to_drop)

# Now I can do the same for NP and VPs
train_np_vp = train_df["noun_verb_phrases"].tolist()
np_vp_to_drop = get_most_and_least_freq_tokens(train_np_vp, 0, BOTTOM_NP_VP_PROPORTION)
train_np_vp_filtered = filter_tokens(train_np_vp, np_vp_to_drop)

# Now combine all remaining tokens
final_train_reviews = [
    tokens + np_vp
    for tokens, np_vp in zip(train_tokens_filtered, train_np_vp_filtered)
]
print(f"First review in train set: {final_train_reviews[0]}")

### --- Tokenising the evaluation dataset --- ###

# Get singular tokens
eval_tokenised_reviews = eval_df[TOKENISATION_TO_USE].tolist()

# Now I can do the same for NP and VPs
eval_np_vp = eval_df["noun_verb_phrases"].tolist()

# Now combine all remaining tokens
final_eval_reviews = [
    tokens + np_vp
    for tokens, np_vp in zip(eval_tokenised_reviews, eval_np_vp)
]
print(f"First review in evaluation set: {final_eval_reviews[0]}")

### --- Get the Vocabulary --- ###

vc = get_vocabulary(final_train_reviews)
print(f"\nThe vocabulary, {len(vc)} in length: ")
print(vc)

18 of the most common tokens being dropped (0.1% of all tokens)
10216 of the least common tokens being dropped (60.0% of all tokens)
Total tokens to drop: 10234
73890 of the least common tokens being dropped (99.5% of all tokens)
Total tokens to drop: 73890
First review in train set: ['think', 'unfortun', 'lead', 'includ', 'word', 'clueless', 'appal', 'nonsens', 'think', 'funny', 'excel', 'entertain', 'suspend', 'disbeliev', 'homosex', 'man', 'lesb', 'wom', 'could', 'fal', 'lov', 'child', 'liv', 'togeth', 'happy', 'alway', 'wond', 'play', 'heart', 'warm', 'imposs', 'far', 'implaus', 'describ', 'script', 'funny', 'neg', 'could', 'wel', 'end', 'famy', 'driv', 'away', 'init', 'hous', 'instead', 'extend', 'expl', 'wheth', 'man', 'retain', 'resid', 'homosex', 'this movie', 'a child', 'a movie', 'other movies', 'the acting', 'the script', 'the story', 'the family', 'the man']
First review in evaluation set: ['feel', 'tot', 'rip', 'someon', 'nee', 'refund', 'spent', 'blockbust', 'rent', 'home

In [ ]:
run_model_predictions(final_train_reviews, final_eval_reviews, vc, False)

Vectorised Train TF-IDFs Shape: (2799, 7165)
Vectorised Evaluation TF-IDFs Shape: (600, 7165)
Accuracy: 0.8166666666666667
Classification Report Summary:
              precision    recall  f1-score   support

           0       0.77      0.89      0.83       300
           1       0.87      0.74      0.80       300

    accuracy                           0.82       600
   macro avg       0.82      0.82      0.82       600
weighted avg       0.82      0.82      0.82       600



This was the best accuracy I achieved with NP and VP in the mix, which did not live up to the accuracies of ngram usage. I will continue to use my best feature set - stemming and bigrams, for use on the test set.

# Test Set Evaluation with Best Features

In [ ]:
### --- Test set tokenisation --- ###

test_df["cleaned_review"] = test_df["raw_review"].apply(clean_text)
test_df["tokenised_by_whitespace"] = test_df["cleaned_review"].apply(tokenise_by_whitespace)
test_df["tokenised_by_stemming"] = test_df["cleaned_review"].apply(tokenise_by_stemming)

### --- Now lets add the bigrams, trigrams, and quadgrams to their own columns for the TEST dataframe --- ###

# Get ngrams
tokenised_reviews = test_df["tokenised_by_whitespace"].tolist()
stopwords_removed = remove_stopwords(tokenised_reviews)

# Add them to the dataframe
test_df['bigrams'] = get_ngrams(stopwords_removed, 2)

test_df.tail()

,raw_review,classification,cleaned_review,tokenised_by_whitespace,tokenised_by_stemming,bigrams
2212,Jennifer Jason Leigh and Mare Winningham are a...,0,jennifer jason leigh and mare winningham are a...,"[jennifer, jason, leigh, and, mare, winningham...","[jen, jason, leigh, mar, winningham, good, mat...","[jennifer jason, jason leigh, leigh mare, mare..."
3136,the cover of the box makes this movie look rea...,0,the cover of the box makes this movie look rea...,"[the, cover, of, the, box, makes, this, movie,...","[cov, box, mak, movy, look, real, good, fool, ...","[cover box, box makes, makes movie, movie look..."
639,Otto Preminger's noir classic works almost as ...,1,otto preminger s noir classic works almost as ...,"[otto, preminger, s, noir, classic, works, alm...","[otto, prem, noir, class, work, almost, flip, ...","[otto preminger, preminger noir, noir classic,..."
2123,The Twilight Zone has achieved a certain mytho...,0,the twilight zone has achieved a certain mytho...,"[the, twilight, zone, has, achieved, a, certai...","[twilight, zon, achiev, certain, mytholog, muc...","[twilight zone, zone achieved, achieved certai..."
112,This is comedy as it once was and comparing th...,1,this is comedy as it once was and comparing th...,"[this, is, comedy, as, it, once, was, and, com...","[comedy, comp, two, remak, money, pit, don, ye...","[comedy comparing, comparing two, two remakes,..."


In [ ]:
# Token top and bottom %s to remove
TOP_TOKEN_PROPORTION = 0.001
BOTTOM_TOKEN_PROPORTION = 0.6
BOTTOM_BIGRAMS_PROPORTION = 0.99
TOKENISATION_TO_USE = "tokenised_by_stemming"

### --- Tokenising and filtering the train dataset --- ###

# Get singular tokens and filter
train_tokenised_reviews = train_df[TOKENISATION_TO_USE].tolist()
tokens_to_drop = get_most_and_least_freq_tokens(train_tokenised_reviews, TOP_TOKEN_PROPORTION, BOTTOM_TOKEN_PROPORTION)
train_tokens_filtered = filter_tokens(train_tokenised_reviews, tokens_to_drop)

# Now I can do the same for bigrams
train_bigrams = train_df["bigrams"].tolist()
bigrams_to_drop = get_most_and_least_freq_tokens(train_bigrams, 0, BOTTOM_BIGRAMS_PROPORTION)
train_bigrams_filtered = filter_tokens(train_bigrams, bigrams_to_drop)

# Now combine all remaining tokens
final_train_reviews = [
    tokens + bigrams
    for tokens, bigrams in zip(train_tokens_filtered, train_bigrams_filtered)
]
print(f"First review in train set: {final_train_reviews[0]}")

### --- Tokenising test set --- ###

# Get singular tokens
test_tokenised_reviews = test_df[TOKENISATION_TO_USE].tolist()

# Same for bigrams
test_bigrams = test_df["bigrams"].tolist()

# Now combine all remaining tokens
final_test_reviews = [
    tokens + bigrams
    for tokens, bigrams in zip(test_tokenised_reviews, test_bigrams)
]
print(f"First review in test set: {final_test_reviews[0]}")

### --- Get the Vocabulary --- ###

vc = get_vocabulary(final_train_reviews)
print(f"\nThe vocabulary, {len(vc)} in length: ")
print(vc)

18 of the most common tokens being dropped (0.1% of all tokens)
10216 of the least common tokens being dropped (60.0% of all tokens)
Total tokens to drop: 10234
273385 of the least common tokens being dropped (99.0% of all tokens)
Total tokens to drop: 273385
First review in train set: ['think', 'unfortun', 'lead', 'includ', 'word', 'clueless', 'appal', 'nonsens', 'think', 'funny', 'excel', 'entertain', 'suspend', 'disbeliev', 'homosex', 'man', 'lesb', 'wom', 'could', 'fal', 'lov', 'child', 'liv', 'togeth', 'happy', 'alway', 'wond', 'play', 'heart', 'warm', 'imposs', 'far', 'implaus', 'describ', 'script', 'funny', 'neg', 'could', 'wel', 'end', 'famy', 'driv', 'away', 'init', 'hous', 'instead', 'extend', 'expl', 'wheth', 'man', 'retain', 'resid', 'homosex', 'funny movie', 'fall love', 'movie one', 'acting good', 'good script', 'story could', 'could well']
First review in test set: ['av', 'gard', 'vary', 'psychot', 'nutcas', 'delicy', 'essay', 'unh', 'gle', 'steph', 'sach', 'knock', 'var

In [ ]:
run_model_predictions(final_train_reviews, final_test_reviews, vc, True)

Vectorised Train TF-IDFs Shape: (2799, 9555)
Vectorised Evaluation TF-IDFs Shape: (600, 9555)
Accuracy: 0.8466666666666667
Classification Report Summary:
              precision    recall  f1-score   support

           0       0.84      0.86      0.85       300
           1       0.85      0.84      0.85       300

    accuracy                           0.85       600
   macro avg       0.85      0.85      0.85       600
weighted avg       0.85      0.85      0.85       600



# Naive Bayes from Scratch

In [ ]:
from collections import defaultdict
import numpy as np

class NaiveBayes():
    """
    An implementation of a Naive Bayes classifier from scratch.
    Intended to be used for a sentiment analysis task.
    """

    def __init__(self, pos_reviews: list, neg_reviews: list, alpha: float = 1.0):
        """
        Initialise the classifier with positive and negative reviews.

        NOTE: these are assumed to already be tokenised.
        """

        # The reviews (already tokenised)
        self.pos_reviews = pos_reviews
        self.neg_reviews = neg_reviews

        # Laplace smoothing parameter (default will be 1.0)
        self.alpha = alpha

        ### --- Required in training and prediction --- ###

        # Will store the vocabulary of words
        self.vocabulary = set()
        self.vocab_size = 0

        # Will store the total number of words that appears in each class
        self.pos_total_words = 0
        self.neg_total_words = 0

        # Will store counts for each word in each class
        self.pos_word_counts = defaultdict(int)
        self.neg_word_counts = defaultdict(int)

        # Class prior probabilities
        self.pos_prior_prob = 0.0
        self.neg_prior_prob = 0.0

        # Will store probabilities for each word appearing in each class
        self.pos_word_probs = {}
        self.neg_word_probs = {}



    def count_word_appearances(self):
        """
        Count the number of appearances of each word in the positive and negative reviews.
        These will be stored in the pos_word_counts and neg_word_counts dictionaries.
        """

        # I am assuming that data here is already tokenised
        for review in self.pos_reviews:
            for token in review:
                self.pos_word_counts[token] += 1

        for review in self.neg_reviews:
            for token in review:
                self.neg_word_counts[token] += 1

    def train(self):
        """
        Train the classifier using the tokenised positive and negative reviews given.
        Performs the following steps:
        - Getting the word counts for positive and negative reviews
        - Produces a vocabulary set filled with all unique words in all reviews
        - Calculates the total number of words in each class
        - Calculates the prior probabilities for each class (expected to both be 0.5 here)
        - Calculates the probability of each word appearing in each class and stores them
          in the pos_word_probs and neg_word_probs dictionaries
        """

        # Get word counts for both positive and negative classes
        self.count_word_appearances()

        # Now produce a vocab
        self.vocabulary = self.pos_word_counts.keys() | self.neg_word_counts.keys()
        self.vocab_size = len(self.vocabulary)

        # Get total word counts for each class
        self.pos_total_words = sum(self.pos_word_counts.values())
        self.neg_total_words = sum(self.neg_word_counts.values())

        # Compute prior probabilities
        # NOTE: I expect they should always be equal for this assignment.
        # I'll compute them anyway.
        total_reviews = len(self.pos_reviews) + len(self.neg_reviews)
        self.pos_prior_prob = len(self.pos_reviews) / total_reviews
        self.neg_prior_prob = len(self.neg_reviews) / total_reviews

        # Calculate prob for each word in the vocabulary.
        # For each class c and word w, it is given by:
        # P(w | c) = (count(word_in_c) + a) / (total_c_word_count + a * vocab_size)
        for word in self.vocabulary:
            self.pos_word_probs[word] = (
                (self.pos_word_counts[word] + self.alpha) /
                (self.pos_total_words + self.alpha * self.vocab_size)
            )
            self.neg_word_probs[word] = (
                (self.neg_word_counts[word] + self.alpha) /
                (self.neg_total_words + self.alpha * self.vocab_size)
            )

        # Let's print the results to double check
        print(f"Vocabulary size: {self.vocab_size}")
        print(f"Total positive words: {self.pos_total_words}")
        print(f"Total negative words: {self.neg_total_words}")

    def make_single_prediction(self, review: list):
        """
        Takes in one tokenised review and will use the pre-computed probailities
        from the training stage to make a prediction on whether a review is positive
        or negative in sentiment.

        Returns 1 for positive, and 0 for negative.

        NOTE: expects that the given review is tokensied in the same way as the
        reviews used to train the model.
        """

        if len(review) == 0:
            raise ValueError("Review cannot be empty!")

        # I'll start with the prior probailities.
        # I am using the log of the probabilities, so I can add instead of multiply.
        # This is because the probabilities will be super small so underflow
        # can occur wih multiplication.
        log_prob_pos = np.log(self.pos_prior_prob)
        log_prob_neg = np.log(self.neg_prior_prob)

        # The default probability for unknown words - precompute to avoid calculation
        # every single time
        default_pos_prob = np.log(
            self.alpha / (self.pos_total_words + self.alpha * self.vocab_size)
        )
        default_neg_prob = np.log(
            self.alpha / (self.neg_total_words + self.alpha * self.vocab_size)
        )

        # Calculate scores
        for word in review:
            if word in self.vocabulary:
                # avoid multiplication of extremely small nums here - add logs instead
                log_prob_pos += np.log(self.pos_word_probs[word])
                log_prob_neg += np.log(self.neg_word_probs[word])
            else:
                # Handle unknown words with small default probability (laplace)
                log_prob_pos += default_pos_prob
                log_prob_neg += default_neg_prob

        # Compare log probability sum results.
        # Return 1 for positive prediction, and 0 for negative prediction.
        if log_prob_pos > log_prob_neg:
            return 1
        else:
            return 0

    def predict(self, reviews: list):
        """
        Runs model predictions for each review in the list of data provided.

        Returns the predictions in a list.
        """

        # Can't make predictions if theres no data
        if len(reviews) == 0:
            raise ValueError("Reviews cannot be empty!")

        predictions = []

        for review in reviews:
            predictions.append(self.make_single_prediction(review))

        return predictions

    def get_accuracy(self, true_labels: list, predicted_labels: list):
        """
        Calculates the accuracy of the model's predictions.
        """

        # Can't compare if they are not equal in length
        if len(true_labels) != len(predicted_labels):
            raise ValueError("True and predicted labels must be equal length!")

        correct = 0
        total = len(true_labels)

        # Now loop through and check which were predicted correctly
        for i in range(total):
            if true_labels[i] == predicted_labels[i]:
                correct += 1

        # Return final accuracy score
        return correct / total

## Use my Naive Bayes to Classify the Test Set

I will be using the same best-performing features from earlier: stemming + ngrams up to and including n=4.

In [ ]:
# Constants for token proportions
TOP_TOKEN_PROPORTION = 0.001
BOTTOM_TOKEN_PROPORTION = 0.6
BOTTOM_BIGRAMS_PROPORTION = 0.99
TOKENISATION_TO_USE = "tokenised_by_stemming"

# Function to filter and separate by classification - I needed to add something to help me split the positive and negative reviews into separate lists,
# because this is what my NaiveBayes classifier requires for its constructor.
def separate_reviews_by_classification(data: list, classifications: list):
    pos_reviews = []
    neg_reviews = []
    for review, classification in zip(data, classifications):
        if classification == 1:
            pos_reviews.append(review)
        else:
            neg_reviews.append(review)
    return pos_reviews, neg_reviews

### --- Tokenising and filtering the train dataset --- ###

# Get singular tokens and filter
train_tokenised_reviews = train_df[TOKENISATION_TO_USE].tolist()
tokens_to_drop = get_most_and_least_freq_tokens(train_tokenised_reviews, TOP_TOKEN_PROPORTION, BOTTOM_TOKEN_PROPORTION)
train_tokens_filtered = filter_tokens(train_tokenised_reviews, tokens_to_drop)

# Now I can do the same for bigrams...
train_bigrams = train_df["bigrams"].tolist()
bigrams_to_drop = get_most_and_least_freq_tokens(train_bigrams, 0, BOTTOM_BIGRAMS_PROPORTION)
train_bigrams_filtered = filter_tokens(train_bigrams, bigrams_to_drop)

# Now combine all remaining tokens
final_train_reviews = [
    tokens + bigrams
    for tokens, bigrams in zip(train_tokens_filtered, train_bigrams_filtered)
]

# Separate into positive and negative reviews based on classification
classifications = train_df["classification"].tolist()
train_pos_reviews, train_neg_reviews = separate_reviews_by_classification(final_train_reviews, classifications)

# Sanity check of the review content. These are what will be passed to my NaiveBayes constructor
print(f"First positive review: {train_pos_reviews[0]}")
print(f"First negative review: {train_neg_reviews[0]}")

### --- Process the test set with the same features --- ###

# Get singular tokens
test_tokenised_reviews = test_df[TOKENISATION_TO_USE].tolist()

# Same for bigrams
test_bigrams = test_df["bigrams"].tolist()

# Now combine all remaining tokens
final_test_reviews = [
    tokens + bigrams
    for tokens, bigrams in zip(test_tokenised_reviews, test_bigrams)
]
print(f"First review in test set: {final_test_reviews[0]}")


18 of the most common tokens being dropped (0.1% of all tokens)
10216 of the least common tokens being dropped (60.0% of all tokens)
Total tokens to drop: 10234
273385 of the least common tokens being dropped (99.0% of all tokens)
Total tokens to drop: 273385
First positive review: ['think', 'unfortun', 'lead', 'includ', 'word', 'clueless', 'appal', 'nonsens', 'think', 'funny', 'excel', 'entertain', 'suspend', 'disbeliev', 'homosex', 'man', 'lesb', 'wom', 'could', 'fal', 'lov', 'child', 'liv', 'togeth', 'happy', 'alway', 'wond', 'play', 'heart', 'warm', 'imposs', 'far', 'implaus', 'describ', 'script', 'funny', 'neg', 'could', 'wel', 'end', 'famy', 'driv', 'away', 'init', 'hous', 'instead', 'extend', 'expl', 'wheth', 'man', 'retain', 'resid', 'homosex', 'funny movie', 'fall love', 'movie one', 'acting good', 'good script', 'story could', 'could well']
First negative review: ['say', 'grainy', 'poor', 'mm', 'stag', 'best', 'attract', 'perform', 'germ', 'noth', 'posit', 'avoid', 'travesty'

In [ ]:
# Now I can pass in my tokenised reviews into my classifier and train it
nb = NaiveBayes(train_pos_reviews, train_neg_reviews)
nb.train()

# Lets show some word probabilities
print(nb.pos_word_probs)
print(nb.neg_word_probs)

Vocabulary size: 9555
Total positive words: 163927
Total negative words: 159462
{'movie plot': 1.7292860354388352e-05, 'mckay': 2.8821433923980587e-05, 'myst': 5.187858106316505e-05, 'geoffrey': 2.8821433923980587e-05, 'bogdanovich': 1.1528573569592235e-05, 'lot fun': 4.611429427836894e-05, 'phrase': 1.1528573569592235e-05, 'secret service': 3.4585720708776704e-05, 'agend': 2.8821433923980587e-05, 'movies time': 2.8821433923980587e-05, 'spectac': 2.8821433923980587e-05, 'sentinel': 3.4585720708776704e-05, 'replet': 1.7292860354388352e-05, 'way make': 2.8821433923980587e-05, 'pretend': 8.646430177194176e-05, 'sure movie': 1.7292860354388352e-05, 'camaradery': 1.7292860354388352e-05, 'trio': 2.8821433923980587e-05, 'rev': 0.0004150286485053204, 'kevin spacey': 1.7292860354388352e-05, 'antonio': 2.305714713918447e-05, 'mind': 0.0007666501423778836, 'kor': 5.187858106316505e-05, 'charit': 1.7292860354388352e-05, 'regurgit': 1.1528573569592235e-05, 'mason': 4.035000749357282e-05, 'bows': 4.

In [ ]:
### --- Now I can take the stemmed test reviews and their classifications, run model predictions, and check my accuracy against the true labels --- ###

# Get the test reviews and their corresponding true labels (to be used in the accuracy check)
test_reviews = test_df['tokenised_by_stemming'].tolist()
true_labels = test_df['classification'].tolist()

# Sanity check
print("First review in the test set:", test_reviews[0])
print("The true labels:", true_labels)

### --- Now to perform predictions, and check the model's accuracy --- ###

predicted_classes = nb.predict(test_reviews)
accuracy = nb.get_accuracy(true_labels, predicted_classes)
print(f"\n\nThe accuracy of the model of Naive Bayes from scratch on the test set is: {accuracy}")

First review in the test set: ['av', 'gard', 'vary', 'psychot', 'nutcas', 'delicy', 'essay', 'unh', 'gle', 'steph', 'sach', 'knock', 'vary', 'dim', 'wit', 'young', 'adult', 'us', 'term', 'loos', 'dayton', 'hal', 'univers', 'clos', 'demolit', 'feat', 'dread', 'act', 'entir', 'cast', 'daphn', 'zunig', 'mak', 'ignominy', 'inauspicy', 'film', 'debut', 'debby', 'bimbo', 'head', 'crush', 'car', 'hefty', 'corps', 'tal', 'okay', 'mak', 'f', 'x', 'matthew', 'mungl', 'bloody', 'murd', 'basebal', 'bat', 'bludgeon', 'chick', 'wir', 'strangulation', 'standard', 'dril', 'head', 'bit', 'sort', 'gruesom', 'thing', 'downb', 'surpr', 'twist', 'end', 'lat', 'cop', 'intrud', 'creepy', 'scor', 'christopher', 'hellra', 'young', 'slight', 'smidg', 'gratuit', 'fem', 'nud', 'endear', 'incompet', 'direct', 'jeffrey', 'obrow', 'stev', 'carp', 'also', 'bless', 'us', 'pow', 'kindr', 'entertain', 'abysm', 'slic', 'n', 'dic', 'atroc', 'siz', 'good', 'deal', 'delect', 'dopey', 'drecky', 'low', 'grad', 'fun']
The true

# Using BERT

In this final section of the notebook I will be using BERT, specifically the base model, to perform the same classification.

NOTE: I recommend connecting to a GPU colab runtime otherwise this takes a longgggg time.

In [ ]:
# Install deps
!pip install git+https://github.com/huggingface/transformers.git datasets evaluate torch scikit-learn

# Clone the transformers repo - that way I can use the script for running the classification
!git clone https://github.com/huggingface/transformers.git

# And cd into the script's directory
%cd transformers/examples/pytorch/text-classification

# And install any reqs I have missed
!pip install -r requirements.txt

# Have added this to avoid an annoying popup message asking for visualised results
import os
os.environ["WANDB_DISABLED"] = "true"

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-3x1wgxsf
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-3x1wgxsf
  Resolved https://github.com/huggingface/transformers.git to commit c8c8dffbe45ebef0a8dba4a51024e5e5e498596b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 

## Send Data to CSVs for the script to use

I have created some CSVs to store the training, evaluation and test data in - the script will use these.

In [ ]:
# Create copies, rename cols, and keep only 'text' and 'label'. This is all BERT looks for in the csvs
train_copy = train_df.copy().rename(columns={"classification": "label", "raw_review": "text"})[["text", "label"]]
eval_copy = eval_df.copy().rename(columns={"classification": "label", "raw_review": "text"})[["text", "label"]]
test_copy = test_df.copy().rename(columns={"classification": "label", "raw_review": "text"})[["text", "label"]]

# Now store them as csv files - no index to avoid adding unnecessary column
train_copy.to_csv("/content/train.csv", index=False)
eval_copy.to_csv("/content/eval.csv", index=False)
test_copy.to_csv("/content/test.csv", index=False)

## Classify using BERT - uncased and cased models

I show two run-throughs of using BERT's cased and uncased models below. I have included the first attempt, and the attempt that followed after I adjusted the hyperparameters when training. Specifically, I saw accuracy improvements after increasing the number of epochs from 3 to 10, and doubling the max sequence length from 128 to 256.


### Uncased

In [ ]:
!python run_classification.py \
    --model_name_or_path bert-base-uncased \
    --train_file /content/train.csv \
    --validation_file /content/eval.csv \
    --test_file /content/test.csv \
    --text_column_name text \
    --label_column_name label \
    --metric_name accuracy \
    --do_train \
    --do_eval \
    --do_predict \
    --max_seq_length 128 \
    --per_device_train_batch_size 16 \
    --learning_rate 2e-5 \
    --num_train_epochs 3 \
    --output_dir ./bert_uncased_output/ \
    --overwrite_output_dir

2024-12-08 17:04:20.102355: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-08 17:04:20.289696: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-08 17:04:20.317720: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-08 17:04:20.387264: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-12-08 17:04:25.175045: W tensorflow/comp

The intial resulting accuracy of BERT's uncased model is 82.5%





### Cased


In [ ]:
!python run_classification.py \
    --model_name_or_path bert-base-cased \
    --train_file /content/train.csv \
    --validation_file /content/eval.csv \
    --test_file /content/test.csv \
    --text_column_name text \
    --label_column_name label \
    --metric_name accuracy \
    --do_train \
    --do_eval \
    --do_predict \
    --max_seq_length 128 \
    --per_device_train_batch_size 16 \
    --learning_rate 2e-5 \
    --num_train_epochs 3 \
    --output_dir ./bert_cased_output/ \
    --overwrite_output_dir

2024-12-08 17:08:21.147656: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-08 17:08:21.218517: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-08 17:08:21.238774: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-08 17:08:21.292255: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-12-08 17:08:25.256580: W tensorflow/comp

The initial resulting accuracy of BERT's cased model is 81.5%

## Results after changing hyperparameters

I changed the number of epochs from 3 to 10, and also increased the doubled the max_seq_length (128 to 256). This gave improved accuracy results.

### Uncased


In [ ]:
!python run_classification.py \
    --model_name_or_path bert-base-uncased \
    --train_file /content/train.csv \
    --validation_file /content/eval.csv \
    --test_file /content/test.csv \
    --text_column_name text \
    --label_column_name label \
    --metric_name accuracy \
    --do_train \
    --do_eval \
    --do_predict \
    --max_seq_length 256 \
    --per_device_train_batch_size 16 \
    --learning_rate 2e-5 \
    --num_train_epochs 10 \
    --output_dir ./bert_uncased_output/ \
    --overwrite_output_dir

2024-12-08 17:17:38.952256: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-08 17:17:38.971729: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-08 17:17:38.977565: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-08 17:17:38.992007: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-12-08 17:17:40.385543: W tensorflow/comp

Uncased using more training epochs (10 instead of 3) improved the accuracy up to 87.16%

### Cased

In [ ]:
!python run_classification.py \
    --model_name_or_path bert-base-cased \
    --train_file /content/train.csv \
    --validation_file /content/eval.csv \
    --test_file /content/test.csv \
    --text_column_name text \
    --label_column_name label \
    --metric_name accuracy \
    --do_train \
    --do_eval \
    --do_predict \
    --max_seq_length 256 \
    --per_device_train_batch_size 16 \
    --learning_rate 2e-5 \
    --num_train_epochs 10 \
    --output_dir ./bert_cased_output/ \
    --overwrite_output_dir

2024-12-08 17:39:25.641483: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-08 17:39:25.661631: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-08 17:39:25.667595: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-08 17:39:25.682470: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-12-08 17:39:26.891290: W tensorflow/comp

Cased using more training epochs (10 instead of 3) improved the accuracy up to 86.66%